# Phase 3 + 4: Image models (Pedestrian in Traffic, BITE308L)

**Before running:** right panel → **Session options**
- **Accelerator:** GPU (T4 x2 or P100)
- **Internet:** ON (needed to download the pretrained weights)
- **Input:** add your `pedestrian-crops` dataset

Then click **Run All**. It takes about 30–45 minutes. At the end, download **`phase3_outputs.zip`** from the **Output** section on the right.

## 1. Find the crops dataset and set up folders

In [ ]:
import glob, os
hits = glob.glob("/kaggle/input/**/crops_index.csv", recursive=True)
assert hits, "crops_index.csv not found - did you add your crops dataset as Input?"
os.environ["PS_CROPS_DIR"] = os.path.dirname(hits[0])
print("Crops folder:", os.environ["PS_CROPS_DIR"])
os.makedirs("image_pipeline", exist_ok=True)
open("image_pipeline/__init__.py", "w").close()
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU accelerator in Session options!")

## 2. Project code (same files as in your laptop folder)

In [ ]:
%%writefile config.py
"""
Central settings for the whole project.
Every script imports paths and constants from here, so nothing is hard-coded twice.
"""
import os
from pathlib import Path

# ---------------------------------------------------------------- paths
ROOT = Path(__file__).resolve().parent

DATA_DIR = ROOT / "data"
WIDERPERSON_DIR = DATA_DIR / "widerperson"      # Images/, Annotations/, train.txt, val.txt
MELBOURNE_DIR = DATA_DIR / "melbourne"          # the hourly counts CSV goes here
EVENTS_CSV = DATA_DIR / "events.csv"            # events calendar (built in Phase 1)
CAMERAS_CSV = DATA_DIR / "cameras.csv"          # camera registry (built in Phase 5)

PROCESSED_DIR = DATA_DIR / "processed"
# person crops, one folder per class (PS_CROPS_DIR lets Kaggle point at its read-only input folder)
CROPS_DIR = Path(os.environ.get("PS_CROPS_DIR", PROCESSED_DIR / "crops"))
FEATURES_DIR = PROCESSED_DIR / "features"       # saved backbone fingerprints (.npy)

MODELS_DIR = ROOT / "saved_models"
RESULTS_DIR = ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"

# ---------------------------------------------------------------- image pipeline
# WiderPerson annotation labels -> our crop classes
#   1 pedestrians, 2 riders, 3 partially-visible persons, 4 ignore regions, 5 crowd
WIDERPERSON_LABELS = {1: "pedestrian", 2: "rider", 3: "partially_visible"}
IMAGE_CLASSES = ["pedestrian", "rider", "partially_visible", "background"]
MIN_CROP_SIZE = 24          # skip boxes smaller than this (pixels, either side)
CROP_PAD = 0.10             # add 10% context around each box
CROP_SAVE_MAX_SIDE = 256    # crops are saved with longest side <= this (saves disk)
VAL_FRACTION_OF_TRAIN = 0.15  # part of WiderPerson train images held out for validation
# max crops per class per split (pedestrians are very common; riders are rare and all kept)
CROP_CAPS = {"train": 6000, "val": 1200, "test": 1500}
BACKGROUND_PER_IMAGE = 2    # random empty patches sampled per image
RANDOM_SEED = 42

# pretrained ImageNet weights from the timm library (each model's own input size is used:
# 224 px for VGG16 / ResNet50 / ViT, 299 px for InceptionV3 / Xception)
BACKBONES = {
    "vgg16": "vgg16.tv_in1k",
    "resnet50": "resnet50.a1_in1k",
    "inception_v3": "inception_v3.tv_in1k",
    "xception": "legacy_xception.tf_in1k",
    "vit": "vit_base_patch16_224.augreg2_in21k_ft_in1k",
}
HYBRIDS = [("vit", "xception"), ("vit", "inception_v3"), ("xception", "inception_v3")]
IMAGE_MODELS_DIR = MODELS_DIR / "image"     # trained heads + feature statistics

# ---------------------------------------------------------------- sensor pipeline
# crowd level for the NEXT hour, by percentile of that sensor's own history
LEVEL_PERCENTILES = {"Low": 0.25, "Medium": 0.75, "High": 0.95}   # above 0.95 = Very High
FOOTFALL_LEVELS = ["Low", "Medium", "High", "Very High"]
TEST_FRACTION = 0.2         # split by DATE: the last 20% of the time span is the test set
# Use post-lockdown data (Melbourne's 2020-21 lockdowns make those years unlike normal life).
# If the file has less than a year after this date, the script falls back to PRE_COVID_WINDOW.
START_DATE = "2022-01-01"
PRE_COVID_WINDOW = ("2014-01-01", "2020-02-29")
MIN_SENSOR_DAYS = 180       # drop sensors with less history than this
LAG_HOURS = [1, 2, 3, 24, 168]   # past counts used as features (168 h = same hour last week)
SENSOR_FEATURES_FILE = PROCESSED_DIR / "sensor_features.parquet"

# ---------------------------------------------------------------- risk rules (tunable)
RISK_RULES = {
    "act_density": 4.0,             # people per m2 -> Act now
    "act_density_tight": 3.0,       # people per m2 ...
    "act_hidden_share": 0.40,       # ... combined with this share partly hidden -> Act now
    "watch_density": 2.0,           # people per m2 -> Watch
    "watch_surge_ratio": 1.5,       # crowd vs normal for that slot -> Watch
}

# ---------------------------------------------------------------- agent
LLM_PROVIDER = "gemini"     # "gemini" or "groq"; API key goes in .env

In [ ]:
%%writefile image_pipeline/img_common.py
"""
Shared image helpers: backbones, preprocessing and the classifier head.
Used by Phase 3 (training) and later by the agent's classify tool, so an image
is always prepared in exactly the same way.
"""
import sys
from pathlib import Path

import torch
import torch.nn as nn
from PIL import Image, ImageOps

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from config import BACKBONES


class PadToSquare:
    """Person crops are tall and thin. A plain centre-crop would cut off heads and feet,
    so the crop is padded to a square first (gray borders), then resized."""

    def __call__(self, img: Image.Image) -> Image.Image:
        w, h = img.size
        side = max(w, h)
        return ImageOps.pad(img, (side, side), color=(124, 116, 104))


def load_backbone(name: str, pretrained: bool = True):
    """Return (model, transform). Model outputs a feature vector (no classification layer)."""
    import timm
    from timm.data import resolve_model_data_config, create_transform
    from torchvision import transforms as T

    model = timm.create_model(BACKBONES[name], pretrained=pretrained, num_classes=0).eval()
    cfg = resolve_model_data_config(model)
    cfg["crop_pct"] = 1.0                          # image is already square: keep all of it
    base = create_transform(**cfg, is_training=False)
    transform = T.Compose([PadToSquare(), base])
    return model, transform


def augment_transform(transform):
    """Training-time copy: horizontal flip + brightness change (as in the methodology)."""
    from torchvision import transforms as T
    return T.Compose([T.RandomHorizontalFlip(p=1.0), T.ColorJitter(brightness=0.3), transform])


class Head(nn.Module):
    """Small classifier on top of frozen features: dense -> ReLU -> dropout -> dense."""

    def __init__(self, in_dim: int, n_classes: int, hidden: int = 512, dropout: float = 0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, n_classes),
        )

    def forward(self, x):
        return self.net(x)


def device():
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
%%writefile image_pipeline/02_extract_features.py
"""
PHASE 3A - Extract "fingerprints" (feature vectors) with 5 pretrained backbones
-------------------------------------------------------------------------------
Each backbone looks at every crop ONCE and turns it into a list of numbers:
    VGG16 4096 | ResNet50 2048 | InceptionV3 2048 | Xception 2048 | ViT 768
These are saved to disk, so all later training takes minutes instead of hours.

The train split is processed twice: the original crops and an augmented copy
(horizontal flip + brightness change), giving the heads more variety to learn from.

Run on a GPU (Kaggle / Colab):   python image_pipeline/02_extract_features.py
Options:  --backbones vit xception     only these backbones
          --no-pretrained              random weights (pipeline test only)
          --limit 200                  only N crops per split (pipeline test only)
"""
import argparse
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from config import CROPS_DIR, FEATURES_DIR, IMAGE_CLASSES, BACKBONES, RANDOM_SEED
from image_pipeline.img_common import load_backbone, augment_transform, device


class Crops(Dataset):
    def __init__(self, paths, transform):
        self.paths, self.transform = list(paths), transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        # crops_index.csv made on Windows uses "\\" - normalise so it also works on Linux (Kaggle)
        with Image.open(CROPS_DIR / self.paths[i].replace("\\", "/")) as im:
            return self.transform(im.convert("RGB"))


@torch.no_grad()
def extract(model, paths, transform, dev, batch=64, workers=2, desc=""):
    loader = DataLoader(Crops(paths, transform), batch_size=batch, num_workers=workers,
                        pin_memory=dev.type == "cuda")
    out = []
    for x in tqdm(loader, desc=desc, leave=False):
        x = x.to(dev, non_blocking=True)
        with torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=dev.type == "cuda"):
            f = model(x)
        out.append(f.float().cpu().numpy())
    return np.concatenate(out)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--backbones", nargs="*", default=list(BACKBONES))
    ap.add_argument("--no-pretrained", action="store_true")
    ap.add_argument("--limit", type=int, default=0)
    ap.add_argument("--workers", type=int, default=2)
    args = ap.parse_args()

    torch.manual_seed(RANDOM_SEED)
    dev = device()
    print(f"Device: {dev}" + ("" if dev.type == "cuda" else "   [WARNING] no GPU - this will be very slow"))
    FEATURES_DIR.mkdir(parents=True, exist_ok=True)

    index = pd.read_csv(CROPS_DIR / "crops_index.csv")
    index["label"] = index["class"].map({c: i for i, c in enumerate(IMAGE_CLASSES)})
    if args.limit:
        index = index.groupby(["split", "class"], group_keys=False).head(args.limit // 4)
    for split in ["train", "val", "test"]:
        part = index[index["split"] == split]
        np.save(FEATURES_DIR / f"labels_{split}.npy", part["label"].to_numpy())
        print(f"{split:<5}: {len(part):>6,} crops  {part['class'].value_counts().to_dict()}")

    for name in args.backbones:
        t0 = time.time()
        model, transform = load_backbone(name, pretrained=not args.no_pretrained)
        model = model.to(dev)
        jobs = [("train", transform), ("train_aug", augment_transform(transform)),
                ("val", transform), ("test", transform)]
        for tag, tf in jobs:
            split = tag.replace("_aug", "")
            paths = index.loc[index["split"] == split, "path"]
            feats = extract(model, paths, tf, dev, workers=args.workers, desc=f"{name}:{tag}")
            np.save(FEATURES_DIR / f"{name}_{tag}.npy", feats)
        print(f"{name:<13} dim {feats.shape[1]:>5}   done in {time.time() - t0:5.0f}s")
        del model
        if dev.type == "cuda":
            torch.cuda.empty_cache()
    print(f"\nFeatures saved in {FEATURES_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile image_pipeline/03_train_heads.py
"""
PHASE 3B + 4 - Train the image classifiers on the saved fingerprints
--------------------------------------------------------------------
  5 single models   : VGG16, ResNet50, InceptionV3, Xception, ViT
  3 hybrid models   : fingerprints of two backbones joined side by side (feature fusion)
                      ViT+Xception, ViT+InceptionV3, Xception+InceptionV3
  1 ensemble        : soft voting = average of the 3 hybrids' probabilities

Every model is the same small head:  dense(512) -> ReLU -> dropout -> dense(4 classes)
Riders are rare, so the loss gives each class equal total weight.
Each head keeps the epoch with the best VALIDATION macro-F1; TEST is only used for the final scores.

Outputs
  saved_models/image/heads.pt                     hybrid heads + feature statistics (used by the agent)
  results/tables/image_model_comparison.csv
  results/tables/image_ensemble_report.txt
  results/figures/image_model_comparison.png, image_confusion_ensemble.png, image_per_class_f1.png

Run (Kaggle, after 02_extract_features.py; also fine on a laptop CPU, ~10 min):
    python image_pipeline/03_train_heads.py
"""
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, \
    precision_recall_fscore_support

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))
from config import (FEATURES_DIR, IMAGE_MODELS_DIR, TABLES_DIR, FIGURES_DIR, IMAGE_CLASSES,
                    BACKBONES, HYBRIDS, RANDOM_SEED)
from image_pipeline.img_common import Head, device

EPOCHS, PATIENCE, BATCH, LR = 40, 6, 256, 1e-3
NAMES = {"vgg16": "VGG16", "resnet50": "ResNet50", "inception_v3": "InceptionV3",
         "xception": "Xception", "vit": "ViT"}


# ======================================================================== data
def load_feats():
    y = {s: np.load(FEATURES_DIR / f"labels_{s}.npy") for s in ["train", "val", "test"]}
    y["train"] = np.concatenate([y["train"], y["train"]])          # original + augmented copy
    X, stats = {}, {}
    for b in BACKBONES:
        tr = np.concatenate([np.load(FEATURES_DIR / f"{b}_train.npy"),
                             np.load(FEATURES_DIR / f"{b}_train_aug.npy")]).astype(np.float32)
        mean, std = tr.mean(0), tr.std(0) + 1e-6                    # statistics from TRAIN only
        stats[b] = (mean, std)
        X[b] = {"train": (tr - mean) / std}
        for s in ["val", "test"]:
            X[b][s] = (np.load(FEATURES_DIR / f"{b}_{s}.npy").astype(np.float32) - mean) / std
    return X, y, stats


def join(X, parts, split):
    return np.concatenate([X[p][split] for p in parts], axis=1)


# ======================================================================== training
def train_head(Xtr, ytr, Xva, yva, dev):
    torch.manual_seed(RANDOM_SEED)
    n_cls = len(IMAGE_CLASSES)
    counts = np.bincount(ytr, minlength=n_cls)
    weights = torch.tensor(len(ytr) / (n_cls * np.maximum(counts, 1)), dtype=torch.float32, device=dev)
    head = Head(Xtr.shape[1], n_cls).to(dev)
    opt = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    loss_fn = nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)
    Xt, yt = torch.tensor(Xtr, device=dev), torch.tensor(ytr, device=dev)
    Xv = torch.tensor(Xva, device=dev)
    best, best_state, bad = -1.0, None, 0
    for epoch in range(EPOCHS):
        head.train()
        perm = torch.randperm(len(Xt), device=dev)
        for i in range(0, len(Xt), BATCH):
            idx = perm[i:i + BATCH]
            if len(idx) < 2:
                continue
            opt.zero_grad()
            loss_fn(head(Xt[idx]), yt[idx]).backward()
            opt.step()
        sched.step()
        f1 = f1_score(yva, predict(head, Xv).argmax(1), average="macro")
        if f1 > best:
            best, best_state, bad = f1, {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    head.load_state_dict(best_state)
    return head.cpu().eval(), best, epoch + 1


@torch.no_grad()
def predict(head, X):
    head.eval()
    if not torch.is_tensor(X):
        X = torch.tensor(X, device=next(head.parameters()).device)
    return torch.softmax(head(X), dim=1).cpu().numpy()


def score(name, kind, y, prob, val_f1, secs, epochs):
    pred = prob.argmax(1)
    p, r, f, _ = precision_recall_fscore_support(y, pred, labels=range(len(IMAGE_CLASSES)), zero_division=0)
    mp, mr, mf, _ = precision_recall_fscore_support(y, pred, average="macro", zero_division=0)
    row = {"model": name, "type": kind, "val_macro_f1": val_f1, "test_accuracy": accuracy_score(y, pred),
           "test_macro_precision": mp, "test_macro_recall": mr, "test_macro_f1": mf,
           "train_seconds": secs, "epochs": epochs}
    for i, c in enumerate(IMAGE_CLASSES):
        row[f"f1_{c}"] = f[i]
    return row


# ======================================================================== main
def main():
    dev = device()
    IMAGE_MODELS_DIR.mkdir(parents=True, exist_ok=True)
    TABLES_DIR.mkdir(parents=True, exist_ok=True)
    FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    X, y, stats = load_feats()
    print(f"Device: {dev} | train {len(y['train']):,} (incl. augmented) | val {len(y['val']):,} | test {len(y['test']):,}")

    rows, probs_val, probs_test, heads = [], {}, {}, {}
    plan = [(NAMES[b], "single", [b]) for b in BACKBONES] + \
           [(f"{NAMES[a]} + {NAMES[b]}", "hybrid", [a, b]) for a, b in HYBRIDS]
    for name, kind, parts in plan:
        t0 = time.time()
        head, val_f1, ep = train_head(join(X, parts, "train"), y["train"], join(X, parts, "val"), y["val"], dev)
        secs = time.time() - t0
        probs_val[name] = predict(head, join(X, parts, "val"))
        probs_test[name] = predict(head, join(X, parts, "test"))
        heads[name] = (head, parts)
        rows.append(score(name, kind, y["test"], probs_test[name], val_f1, secs, ep))
        print(f"{name:<26} {kind:<7} val macro-F1 {val_f1:.3f}  test macro-F1 {rows[-1]['test_macro_f1']:.3f}  ({secs:.0f}s, {ep} epochs)")

    hybrid_names = [n for n, k, _ in plan if k == "hybrid"]
    ens_val = np.mean([probs_val[n] for n in hybrid_names], axis=0)
    ens_test = np.mean([probs_test[n] for n in hybrid_names], axis=0)
    ens_name = "Soft-voting ensemble (3 hybrids)"
    rows.append(score(ens_name, "ensemble", y["test"], ens_test,
                      f1_score(y["val"], ens_val.argmax(1), average="macro"), 0.0, 0))
    print(f"{ens_name:<26} val macro-F1 {rows[-1]['val_macro_f1']:.3f}  test macro-F1 {rows[-1]['test_macro_f1']:.3f}")

    table = pd.DataFrame(rows).set_index("model").round(4)
    table.to_csv(TABLES_DIR / "image_model_comparison.csv")
    pd.set_option("display.width", 220)
    print("\n" + table.drop(columns=["train_seconds", "epochs"]).to_string())

    report = classification_report(y["test"], ens_test.argmax(1), target_names=IMAGE_CLASSES, digits=3, zero_division=0)
    (TABLES_DIR / "image_ensemble_report.txt").write_text(f"{ens_name}\n\n{report}", encoding="utf-8")
    print(f"\n{ens_name} on TEST:\n{report}")

    # ---- save what the live system needs: the 3 hybrid heads + feature statistics
    used = sorted({p for n in hybrid_names for p in heads[n][1]})
    torch.save({
        "classes": IMAGE_CLASSES,
        "backbones": {b: BACKBONES[b] for b in used},
        "stats": {b: (torch.tensor(stats[b][0]), torch.tensor(stats[b][1])) for b in used},
        "heads": {n: {"parts": heads[n][1], "in_dim": int(sum(stats[p][0].shape[0] for p in heads[n][1])),
                      "state_dict": heads[n][0].state_dict()} for n in hybrid_names},
    }, IMAGE_MODELS_DIR / "heads.pt")
    print(f"Saved: {IMAGE_MODELS_DIR / 'heads.pt'}")

    make_figures(table, confusion_matrix(y["test"], ens_test.argmax(1), labels=range(len(IMAGE_CLASSES))), ens_name)


# ======================================================================== figures
def make_figures(table, cm, ens_name):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
    BLUE, AQUA, ORANGE = "#2a78d6", "#1baf7a", "#eb6834"
    kind_color = {"single": BLUE, "hybrid": AQUA, "ensemble": ORANGE}
    plt.rcParams.update({"font.size": 10, "axes.edgecolor": GRID, "axes.labelcolor": MUTED,
                         "xtick.color": MUTED, "ytick.color": MUTED, "axes.titlecolor": INK,
                         "axes.spines.top": False, "axes.spines.right": False})

    # 1) macro-F1 for all 9 models, grouped by type
    t = table.iloc[::-1]
    lo = max(0.0, float(t["test_macro_f1"].min()) - 0.1)
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(t) + 1.4))
    yy = np.arange(len(t))
    ax.barh(yy, t["test_macro_f1"], color=[kind_color[k] for k in t["type"]], height=0.62)
    for yi, v in zip(yy, t["test_macro_f1"]):
        ax.text(v + 0.003, yi, f"{v:.3f}", va="center", fontsize=9, color=INK)
    ax.set_yticks(yy, t.index)
    ax.set_xlim(lo, 1.0)
    ax.set_xlabel("macro F1 on test crops")
    ax.grid(axis="x", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in kind_color.values()]
    ax.legend(handles, ["Single backbone", "Hybrid (feature fusion)", "Soft-voting ensemble"],
              frameon=False, loc="lower right")
    ax.set_title("Image models: single vs hybrid vs ensemble", loc="left")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "image_model_comparison.png", dpi=200)
    plt.close(fig)

    # 2) confusion matrix of the ensemble
    n = len(IMAGE_CLASSES)
    cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    labels = [c.replace("_", "\n") for c in IMAGE_CLASSES]
    fig, ax = plt.subplots(figsize=(5.6, 4.8))
    ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{cmn[i, j]:.0%}\n({cm[i, j]:,})", ha="center", va="center", fontsize=8,
                    color="white" if cmn[i, j] > 0.55 else INK)
    ax.set_xticks(range(n), labels)
    ax.set_yticks(range(n), labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title("Confusion matrix on test crops: ensemble", loc="left")
    for s in ax.spines.values():
        s.set_visible(False)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "image_confusion_ensemble.png", dpi=200)
    plt.close(fig)

    # 3) per-class F1: best single vs best hybrid vs ensemble
    singles = table[table["type"] == "single"]
    hybrids = table[table["type"] == "hybrid"]
    pick = [singles["val_macro_f1"].idxmax(), hybrids["val_macro_f1"].idxmax(), ens_name]
    cols = [f"f1_{c}" for c in IMAGE_CLASSES]
    fig, ax = plt.subplots(figsize=(8.5, 4))
    x = np.arange(n)
    w = 0.26
    for i, (m, c) in enumerate(zip(pick, [BLUE, AQUA, ORANGE])):
        vals = table.loc[m, cols].astype(float).to_numpy()
        bars = ax.bar(x + (i - 1) * w, vals, w - 0.03, color=c, label=f"{m}")
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=8, color=INK)
    ax.set_xticks(x, [c.replace("_", " ") for c in IMAGE_CLASSES])
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("F1 on test crops")
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend(frameon=False, loc="upper center", ncol=3, fontsize=8.5, bbox_to_anchor=(0.5, 1.0))
    ax.set_title("Per-class F1: best single vs best hybrid vs ensemble", loc="left")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / "image_per_class_f1.png", dpi=200)
    plt.close(fig)
    print(f"Figures saved in {FIGURES_DIR}")


if __name__ == "__main__":
    main()

## 3. Extract features with the 5 pretrained backbones (~20–35 min)

In [ ]:
!python image_pipeline/02_extract_features.py --workers 4

## 4. Train 5 single models, 3 hybrids and the soft-voting ensemble (~3–5 min)

In [ ]:
!python image_pipeline/03_train_heads.py

## 5. Look at the charts

In [ ]:
from IPython.display import Image, display
for f in ["image_model_comparison", "image_per_class_f1", "image_confusion_ensemble"]:
    display(Image(f"results/figures/{f}.png", width=750))

## 6. Package the results → download `phase3_outputs.zip` from the Output panel

In [ ]:
import shutil, os
os.makedirs("phase3_outputs/saved_models", exist_ok=True)
shutil.copytree("saved_models/image", "phase3_outputs/saved_models/image", dirs_exist_ok=True)
shutil.copytree("results", "phase3_outputs/results", dirs_exist_ok=True)
shutil.make_archive("phase3_outputs", "zip", "phase3_outputs")
print("Ready:", os.path.getsize("phase3_outputs.zip") // 1024, "KB  ->  phase3_outputs.zip")